<a href="https://colab.research.google.com/github/aiyman14/Sch-Mgmt-661-Applications-of-AI-Models/blob/main/W4_Binary_Classification_Classifying_Movie_Reviews_shared.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SCH-MGMT 661: Applications of AI Models  
**Instructor:** Indika Dissanayake  

---

### Tutorial 3: Binary Classification with Neural Networks (using Keras Sequentail Model)

Example adopted from Chapter 4 of Chollet, F. (2023). Deep Learning with Python (3rd Edition). Manning Publications.

---


**Dataset:** IMDb Movie Reviews Dataset

- 50,000 movie reviews from IMDb.
- Already preprocessed: each review is a list of integers (word indices).
- Reviews are split into:
  - **Training set**: 25,000 reviews (50% positive, 50% negative)
  - **Test set**: 25,000 reviews (50% positive, 50% negative)
- Labels:
  - `0` = Negative review  
  - `1` = Positive review

We’ll limit to the top 10,000 most frequent words to keep things manageable.


#### **Load and Explore the dataset**

* Explore the shape of the train and test dataset

It’s always good practice to check the dimensions of the training and test sets:
How many reviews are in the training set?
How many in the test set?
Do the labels match up in size?
This quick check ensures that the dataset was loaded properly.

* Examine a sample review

If we print one review directly, we’ll see a list of integers like:
[1, 14, 22, 16, 43, 530, 973, ...]
This is because the text has already been tokenized into word IDs.

* Decode a sample review:

To make sense of the numbers, we can build a reverse dictionary (index_word) that maps integers back to words. Using this, we can “decode” one review and see the actual text (e.g., “this film was just brilliant casting…”).
This helps us sanity-check the dataset and better understand what the model will learn from.


In [1]:
from tensorflow import keras
from tensorflow.keras.datasets import imdb

# Load the dataset with the top 10,000 most frequent words
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(
    num_words=10000
)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
# shape of the train and test dataset

print(train_data.shape)
print(test_data.shape)
print(train_labels.shape)
print(test_labels.shape)


(25000,)
(25000,)
(25000,)
(25000,)


In [3]:
# Print first review

print(f"First review (word indices):\n{train_data[0]}")

# Corresponding label (1 = positive, 0 = negative)
print(f"Corresponding Label:{train_labels[0]}")


First review (word indices):
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
Corresponding Label:1


In [4]:
# Decode a Review

# Get word index (word → integer)
word_index = imdb.get_word_index()

# Reverse it (integer → word)
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

# Decode the first review by shifting each index by -3 (to skip reserved tokens)
decoded_review = " ".join([reverse_word_index.get(i-3, "?") for i in train_data[0]])

print(decoded_review)


1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should

### Prepare the Data

Neural networks can't work directly with sequences of varying lengths. One way to prepare our data is **multi-hot encoding** — we convert each sequence into a fixed-size vector where:

- The length of the vector is the vocabulary size (10,000 in this case).
- Each index is `1.0` if the word is present in the review, otherwise `0.0`.

This results in a **binary bag-of-words representation**, ignoring word order but preserving word presence.


In [5]:
import numpy as np

# Function to multi-hot encode each sequence

def multi_hot_encode(sequences, num_classes):
    results = np.zeros((len(sequences), num_classes))
    for i, sequence in enumerate(sequences):
        results[i][sequence] = 1.0
    return results

# Apply encoding to training and test data
x_train = multi_hot_encode(train_data, num_classes=10000)
x_test = multi_hot_encode(test_data, num_classes=10000)

# Preview the first encoded review
x_train[0]

array([0., 1., 1., ..., 0., 0., 0.])

In [6]:
# convert labels (`train_labels` and `test_labels`) from integers to `float32` for binary classification with Keras

y_train = train_labels.astype("float32")
y_test = test_labels.astype("float32")

### Model Building, Compilation, and Training

**Build a Sequential model with:**

* Two hidden layers with 16 units and ReLU activation.

* A final output layer with 1 unit and a sigmoid activation for binary classification.

**Compile the Model:**

* Adam optimizer for efficient gradient descent.

* Binary crossentropy as the loss function (since it’s a binary classification task).

* Accuracy as our evaluation metric.

**Train the Model:**

* Train the model for 20 epochs with a batch size of 512.
* Set aside 20% of the training data for validation using validation_split.


**Exercise 1: Build the Model**

👉 Fill in the missing layers.

Input layer: vector of length 10,000

Two hidden layers: 16 units each, with ReLU activation

Output layer: 1 unit, with sigmoid activation

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Normalization, Input

# Build the model

# Build the model
model = Sequential([
    Input(shape=(____,)),         # 👉 fill in
    Dense(____, activation="____"),  # 👉 fill in
    Dense(____, activation="____"),  # 👉 fill in
    Dense(____, activation="____")   # 👉 fill in
])

NameError: name '____' is not defined

**Exercise 2: Compile the Model**

👉 Use the following configuration:

Optimizer: adam

Loss: binary_crossentropy

Metric: accuracy

In [ ]:
# Compile the model
model.compile(
    optimizer="____",        # 👉 fill in
    loss="____",             # 👉 fill in
    metrics=["____"]         # 👉 fill in
)


**Exercise 3: Train the Model**

👉 Train your model using:

20 epochs

batch size of 512

validation split of 20%

In [ ]:
# Train the model
history = model.fit(
    x_train,
    y_train,
    epochs=____,             # 👉 fill in
    batch_size=____,         # 👉 fill in
    validation_split=____,   # 👉 fill in
    verbose=1
)


### Training and Validation Performance: Loss and Accuracy

After training the model, it's important to visualize how well it's learning over time. We'll look at:

- **Training vs. Validation Loss**
- **Training vs. Validation Accuracy**

These plots help us detect **overfitting**, where the model performs well on training data but poorly on unseen data.


In [ ]:
import matplotlib.pyplot as plt

history_dict = history.history
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "r--", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("[IMDB] Training and validation loss")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.clf()
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
plt.plot(epochs, acc, "r--", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("[IMDB] Training and validation accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### Making Predictions with the Trained Model

Once training is complete, we can use the model to generate predictions on unseen data (e.g., `x_test`):

```python
predictions = model.predict(x_test)


In [ ]:
import numpy as np

# Add an extra batch dimension using np.expand_dims
sample = np.expand_dims(x_test[0], axis=0)

# Predict
prediction = model.predict(sample)

decoded_review_test0 = " ".join([reverse_word_index.get(i - 3, "?") for i in test_data[0]])

print(f"Review text:\n{decoded_review_test0}\n")
print(f"Prediction score: {prediction[0][0]:.4f}")
print("Predicted label:", "Positive" if prediction[0][0] > 0.5 else "Negative")
